In [1]:
import argparse
import os
import numpy as np
import math
import sys

import torchvision.transforms as transforms
from torchvision.utils import save_image

from torch.utils.data import DataLoader, Dataset
from PIL import Image
from torchvision import datasets
from torch.autograd import Variable

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as autograd

In [4]:
import torch
import torch_xla.core.xla_model as xm
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [5]:
import torch
import torch_xla.core.xla_model as xm

# Function to check and initialize TPU
def initialize_tpu():
    try:
        # Try to detect TPU device
        device = xm.xla_device()
        print(f"TPU device found: {device}")
        return device
    except ValueError:
        print("No TPU found. Please check your runtime settings and try again.")
        return None

# Initialize TPU
device = initialize_tpu()

# If TPU is initialized successfully, you can now use it for your model and tensors
if device:
    # Example of tensor allocation on TPU
    tensor = torch.randn(2, 2).to(device)
    print("Tensor on TPU:", tensor)


TPU device found: xla:0
Tensor on TPU: tensor([[-1.1677,  0.1786],
        [ 0.0427, -0.0138]], device='xla:0')


In [7]:
# import torch
# print("GPU available:", torch.cuda.is_available())
# print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [8]:
# device = torch.device("cuda" if torch.cuda.is_available() else "gpu")

In [9]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rumiyyaalili","key":"5a48756827bd010b7b5a1f5561da34e8"}'}

In [10]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [11]:
!kaggle datasets download -d rumiyyaalili/brst-dataset-binary

Dataset URL: https://www.kaggle.com/datasets/rumiyyaalili/brst-dataset-binary
License(s): unknown


In [12]:
!unzip -qq brst-dataset-binary

In [13]:
# Create directories
os.makedirs("images", exist_ok=True)

In [14]:
!pip install tfrecord

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 1.3 MB/s eta 0:00:00
  Created wheel for tfrecord: filename=tfrecord-1.14.5-py3-none-any.whl size=14987 sha256=ba0c21d09a624db7341d09b280b2d0ac934a40075d09ab9d2aa36f459557405d
  Stored in directory: /root/.cache/pip/wheels/0c/46/0d/2e6f58f41c2d778dfe3bbb0051a01818af85ce49b82094bc5d
Successfully built tfrecord


In [23]:
import tfrecord
from tfrecord.torch.dataset import TFRecordDataset

tfrecord_path = "/content/brst_dataset.tfrecord"

# Load a sample record to inspect available keys
dataset = TFRecordDataset(tfrecord_path, index_path=None)

# Print the keys of the first record
for record in dataset:
    print("Available keys:", record.keys())  # Display the actual keys
    print("Sample record:", record)
    break  # Only inspect one record

Available keys: dict_keys(['image_raw'])
Sample record: {'image_raw': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\x00\x00\x00\x01\x00\x08\x00\x00\x00\x00y\x19\xf7\xba\x00\x00;\xf8IDATx\x9c\xe5\xbdY\xcc\xaeYv\xdf\xb5\xa6\xbd\x9f\xe1\x1d\xbf\xe1\xcc\xe7\xd4\xa9\xb9k\xean\xb7\xdbm;\xddI:\xc1\x19L\x90b\x04Q\x82\x84\xa2\x90+$\xa4HD\xe4\n\t\x85\\\x00\x12BB(\x97\xdc\x10\x85I\x16\x84\x10)$1N\x10\x02\xe3\xc4\tq\xdb\xddn\xbbz\xa8\xaa\xaeSu\xc6o|\xa7\xe7\xd9\xc3Z\x8b\x8b\xea\x18\x93\x10\x13w\xe8\xae\xf7\xeb\xfa_\xbd\xaf>\xe9\xd1\xfb\xfc\xbe\xb5\xf7Z{\xed\xbd\xd6\x06\xf8\x84\x0b?\xee\x1f\xf0;\xd0-\x19\x9aJN\n\x90\xe0\x9aL\xacYGL\xa5c\xf0\xb0\x81~GR`RiR^\xb4\xa7\xe1N\x9b`i\xb7\xd7w\xbeq\x19\xa7\xb3\xd2>j\xa6\xf4\xfc%\x1cl|\xfa\xf8\xd5\x0f\xaf=\x9a\xbe\xf6\xf5\x1b\xd7\xba\xc9\xe2*\x01\xb8\xd6\x14R\x82P\xf4\xf0Y\xed\'M\xd0.\xd5\x1a\xac\xf8,\x8c\xe3\xa44\xd9Y\x89\xa4\xd1Y?\xd0+72\x9fM\x07\xaf\xddz\xb1\xed\xe8\xe8$\xde~\xfa\xdcE\x83\x07\xbb\xa6\xf6\x03\x9e\xdf\xbb\xb9\xbb\xd5Rw\x95\x00\xf4\xd7a\x

In [24]:
import tfrecord
from tfrecord.torch.dataset import TFRecordDataset

tfrecord_path = "/content/brst_dataset.tfrecord"

# Define the correct feature description
description = {
    "image_raw": "byte"  # Only keep the existing key
}

# Load the dataset
dataset = TFRecordDataset(tfrecord_path, index_path=None, description=description)

# Inspect the first record
for record in dataset:
    print("Image Data:", record["image_raw"][:100])  # Print first 100 bytes for verification
    break  # Take only one record

Image Data: b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\x00\x00\x00\x01\x00\x08\x00\x00\x00\x00y\x19\xf7\xba\x00\x00;\xf8IDATx\x9c\xe5\xbdY\xcc\xaeYv\xdf\xb5\xa6\xbd\x9f\xe1\x1d\xbf\xe1\xcc\xe7\xd4\xa9\xb9k\xean\xb7\xdbm;\xddI:\xc1\x19L\x90b\x04Q\x82\x84\xa2\x90+$\xa4HD\xe4\n\t\x85\\\x00\x12BB('


In [31]:
import torch
from torch.utils.data import IterableDataset, DataLoader
from tfrecord.torch.dataset import TFRecordDataset
from PIL import Image
import io

# Define the feature description (ensure keys match your dataset)
description = {
    "image_raw": "byte"  # Available key from your dataset
}

# Create an iterable PyTorch dataset
class TFRecordIterableDataset(IterableDataset):
    def __init__(self, tfrecord_file, image_size=(256, 256)):
        self.tfrecord_file = tfrecord_file
        self.image_size = image_size  # Resize image to this size

    def parse_record(self, record):
        """Convert raw record into a PyTorch tensor"""
        image_data = record["image_raw"]  # Extract raw image bytes
        image = Image.open(io.BytesIO(image_data))  # Open image from bytes
        image = image.resize(self.image_size)  # Resize image
        image_tensor = torch.tensor(list(image.getdata()), dtype=torch.uint8)  # Convert to tensor
        image_tensor = image_tensor.view(1, self.image_size[0], self.image_size[1])  # Reshape (C, H, W)
        return image_tensor

    def __iter__(self):
        dataset = TFRecordDataset(self.tfrecord_file, index_path=None, description=description)
        for record in dataset:
            yield self.parse_record(record)  # Yield parsed records

# Load dataset function
def load_dataset(batch_size, image_size=(256, 256)):
    tfrecord_file = "/content/brst_dataset.tfrecord"
    dataset = TFRecordIterableDataset(tfrecord_file, image_size=image_size)

    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=2, drop_last=True
    )

    return dataloader

# Example usage
batch_size = 32
dataloader = load_dataset(batch_size, image_size=(256, 256))

# Iterate through the dataset
for batch in dataloader:
    print(batch.shape)  # Check batch shape
    break

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af0130b8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7af0130b8c20>if w.is_alive():

 Traceback (most recent call last):
   File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
       self._shutdown_workers() 
   File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
^^    ^if w.is_alive():^
 ^ ^ ^ ^ ^ ^ ^^^^
^  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^ ^ ^ ^ 
   File "/usr/lib

torch.Size([32, 1, 256, 256])


In [ ]:
import os

dataset_path = "/content"
extensions = (".png", ".jpg", ".jpeg")

# Count all image files recursively
total_images = 0
for root, _, files in os.walk(dataset_path):
    total_images += sum(file.lower().endswith(extensions) for file in files)

print(f"Total number of images in the dataset: {total_images}")

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

dataset_path = "/content"
extensions = (".png", ".jpg", ".jpeg")

# Get the first 50 image file paths
image_files = []
for root, _, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(extensions):
            image_files.append(os.path.join(root, file))
        if len(image_files) >= 50:  # Stop after collecting 50 images
            break
    if len(image_files) >= 50:
        break

# Display the first 50 images
fig, axes = plt.subplots(5, 10, figsize=(15, 10))  # Create a 5x10 grid
axes = axes.flatten()

for img_path, ax in zip(image_files, axes):
    img = Image.open(img_path).convert("L")  # Convert to grayscale
    ax.imshow(img, cmap="gray")  # Display as grayscale
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Set parameters
n_epochs = 150  # Number of epochs of training
batch_size = 64  # Size of the batches
lr_g = 0.0002  # Adam: learning rate for generator
lr_d = 0.00005 # learning rate for discriminator was 0.001
b1 = 0.9  # Adam: decay of first-order momentum of gradient
b2 = 0.999  # Adam: decay of second-order momentum of gradient
latent_dim = 200  # Dimensionality of the latent space was 256
img_size = 256  # Size of each image dimension
channels = 1  # Number of image channels (grayscale)
n_critic = 5  # Number of training steps for discriminator per iteration reduced to 5
#clip_value = 0.01  # Lower and upper clip values for discriminator weights
sample_interval = 400  # Interval between image samples
lambda_gp = 10.0 # Define the gradient penalty weight

In [ ]:
img_shape = (channels, img_size, img_size)
cuda = True if torch.cuda.is_available() else False

In [ ]:
import torch.nn as nn
import numpy as np

class Generator(nn.Module):
    def __init__(self, latent_dim, img_shape):
        super(Generator, self).__init__()
        self.img_shape = img_shape  # Expected shape: (channels, height, width)
        self.channels = img_shape[0]  # Extract channels

        def block(in_feat, out_feat, normalize=True):
            layers = [nn.Linear(in_feat, out_feat)]
            if normalize:
                layers.append(nn.BatchNorm1d(out_feat, 0.8))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        # Fully connected part
        self.model_fc = nn.Sequential(
            *block(latent_dim, 128, normalize=False),
            *block(128, 256),
            *block(256, 512),
            *block(512, 1024),
            nn.Linear(1024, int(np.prod(img_shape))),
            nn.Tanh()
        )

        # Convolutional refinement part
        self.conv_refine = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        # Generate image from latent vector
        img = self.model_fc(z)

        # Ensure correct shape: (batch_size, channels, height, width)
        img = img.view(img.shape[0], self.channels, self.img_shape[1], self.img_shape[2])

        # Apply CNN refinement
        img = self.conv_refine(img)

        return img

In [ ]:
import torch
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self, img_shape):
        super(Discriminator, self).__init__()
        self.img_shape = img_shape  # (channels, height, width)

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Dynamically calculate the flattened feature size
        self.flattened_size = self._get_flattened_size()

        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1)  # Output a single value (real/fake)
        )

    def _get_flattened_size(self):
        """ Forward pass a dummy tensor to compute the flattened size dynamically. """
        with torch.no_grad():
            dummy_input = torch.zeros(1, *self.img_shape)  # Shape: (1, C, H, W)
            dummy_out = self.conv_layers(dummy_input)
            return int(torch.prod(torch.tensor(dummy_out.shape[1:])))  # Compute flattened size

    def forward(self, img):
        features = self.conv_layers(img)  # Apply convolutional layers
        features_flat = features.view(features.shape[0], -1)  # Flatten
        validity = self.fc_layers(features_flat)  # Fully connected layers
        return validity

In [ ]:
def count_parameters_per_layer(model, model_name):
    print(f"\n{model_name} Parameters:")
    for name, param in model.named_parameters():
        print(f"{name}: {param.numel()} parameters")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters in {model_name}: {total_params}")

# Initialize models
generator = Generator(latent_dim, img_shape)
discriminator = Discriminator(img_shape)

# Count parameters per layer
count_parameters_per_layer(generator, "Generator")
count_parameters_per_layer(discriminator, "Discriminator")

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Directory of the dataset
dataset_dir = "/content/InBreast_Aligned_Images"

# Custom Dataset Class to load images with on-the-fly augmentations
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.png')]  # Filter PNG files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_name).convert('L')  # Convert to Grayscale (1 channel)

        if self.transform:
            image = self.transform(image)

        return image

# Define on-the-fly augmentation pipeline
transform = transforms.Compose([
    #transforms.RandomRotation(degrees=15),  # Randomly rotate between -15° and 15°
    #transforms.CenterCrop(224),  # Center crop
    transforms.Resize(256),  # Resize back to 256x256
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize([0.5], [0.5])  # Normalize grayscale images to [-1, 1]
])

# Create Dataset and DataLoader
dataset = CustomImageDataset(root_dir=dataset_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Test one batch to check transformations
for batch in dataloader:
    print(batch.shape)  # Expected: (32, 1, 256, 256) for grayscale images
    break

In [ ]:
import matplotlib.pyplot as plt
import torch

# Function to unnormalize image
def unnormalize(tensor, mean, std):
    """
    Unnormalize the tensor to the range [0, 1].
    """
    mean = torch.tensor(mean).view(-1, 1, 1)
    std = torch.tensor(std).view(-1, 1, 1)
    return tensor * std + mean

# Visualizing one batch of images
def visualize_batch(dataloader):
    # Get one batch from the dataloader
    batch = next(iter(dataloader))  # Get one batch

    # Handle different dataloader formats (with/without labels)
    if isinstance(batch, (list, tuple)):
        images = batch[0]  # Extract images if labels exist
    else:
        images = batch  # If there are no labels, the batch is just images

    # Unnormalize the images (since you used Normalize([0.5], [0.5]))
    images = unnormalize(images, mean=[0.5], std=[0.5])

    # Plot the images in the batch
    batch_size = images.size(0)
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))  # Adjust grid size to display 32 images (4x8 grid)

    for i in range(min(batch_size, 32)):  # Limit to 32 images for display
        ax = axes[i // 8, i % 8]  # Find the correct subplot position
        ax.imshow(images[i].squeeze(), cmap='gray')  # Remove the channel dimension and display as grayscale
        ax.axis('off')  # Hide axes

    plt.show()

# Call function to visualize batch (make sure 'dataloader' is defined)
visualize_batch(dataloader)

In [ ]:
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import torch

def generate_and_display_images(generator, num_images=25, latent_dim=100, nrow=5, device="cuda"):
    """
    Generate and display images using the trained generator.

    Args:
        generator (nn.Module): The trained generator model.
        num_images (int): Number of images to generate.
        latent_dim (int): Dimensionality of the latent space.
        nrow (int): Number of images per row in the grid.
        device (str): Device to perform computations on ("cuda" or "cpu").
    """
    generator.eval()  # Set generator to evaluation mode
    with torch.no_grad():  # No need to compute gradients
        # Generate latent vectors
        z = torch.randn(num_images, latent_dim).to(device)
        # Generate images
        generated_images = generator(z)

        # Debug: Print shape of generated images
        print("Generated images shape:", generated_images.shape)

        # Remove any extra dimensions if necessary
        if generated_images.dim() == 5:  # Check if there's an extra dimension
            generated_images = generated_images.squeeze(2)

        # Normalize and move to CPU for display
        generated_images = generated_images.cpu()

        # Display images using a grid
        grid = make_grid(generated_images, nrow=nrow, normalize=True)
        plt.figure(figsize=(10, 10))
        plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
        plt.axis("off")
        plt.show()

In [ ]:
import torch
from torch.autograd import Variable
import numpy as np
import os
from torchvision.utils import save_image

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define directories inside Google Drive
base_image_dir = "/content/drive/My Drive/GAN_Generated_Images"
save_dir = "/content/drive/My Drive/GAN_Weights"
os.makedirs(base_image_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

checkpoint_interval = 10  # Save images every 10 epochs

# Set the device to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Image shape and model setup
img_shape = (1, img_size, img_size)

# Instantiate models and move them to the GPU
generator = Generator(latent_dim=latent_dim, img_shape=img_shape).to(device)
discriminator = Discriminator(img_shape=img_shape).to(device)

# Define optimizers
optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr_g, betas=(b1, b2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr_d, betas=(b1, b2))

# Learning Rate Scheduler
scheduler_G = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_G, T_max=n_epochs)
scheduler_D = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_D, T_max=n_epochs)

# Gradient penalty function
def compute_gradient_penalty(D, real_samples, fake_samples):
    alpha = torch.tensor(np.random.random((real_samples.size(0), 1, 1, 1)), dtype=torch.float32, device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = D(interpolates)
    fake = Variable(torch.ones(real_samples.shape[0], 1, dtype=torch.float32, device=device), requires_grad=False)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

# ----------
#  Training
# ----------
batches_done = 0
d_loss_val = 0
g_loss_val = 0

for epoch in range(1, n_epochs + 1):
    if epoch % 5 == 1:
        generate_and_display_images(generator, num_images=7, latent_dim=latent_dim, nrow=7, device=device)

    for i, imgs in enumerate(dataloader):
        real_imgs = Variable(imgs.type(torch.float32).to(device))

        # Train Discriminator
        optimizer_D.zero_grad()
        z = Variable(torch.tensor(np.random.normal(0, 1, (imgs.shape[0], latent_dim)), dtype=torch.float32).to(device))
        fake_imgs = generator(z).view(imgs.shape[0], 1, img_size, img_size)
        real_validity = discriminator(real_imgs)
        fake_validity = discriminator(fake_imgs)
        gradient_penalty = compute_gradient_penalty(discriminator, real_imgs.data, fake_imgs.data)
        d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
        d_loss.backward()
        d_loss_val = d_loss.item()
        optimizer_D.step()

        optimizer_G.zero_grad()

        # Train the Generator every n_critic steps
        if i % n_critic == 0:
            fake_imgs = generator(z).view(imgs.shape[0], 1, img_size, img_size)
            fake_validity = discriminator(fake_imgs)
            g_loss = -torch.mean(fake_validity)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
            g_loss.backward()
            optimizer_G.step()
            g_loss_val = g_loss.item()
            if batches_done % sample_interval == 0:
                save_image(fake_imgs.data[:25], "images/%d.png" % batches_done, nrow=5, normalize=True)
            batches_done += 1

        # Step the schedulers (learning rate adjustment)
        scheduler_G.step()
        scheduler_D.step()

    print(f"[Epoch {epoch}/{n_epochs}] [D loss: {d_loss_val}] [G loss: {g_loss_val}]")

    # ----------------------------
    #  Save Images Every 10 Epochs
    # ----------------------------
    if epoch % checkpoint_interval == 0:
        epoch_folder = os.path.join(base_image_dir, f"{epoch}th_epoch")
        os.makedirs(epoch_folder, exist_ok=True)  # Create folder for epoch

        num_samples = 25  # Number of images to save per epoch
        z = torch.randn(num_samples, latent_dim, device=device)
        generated_images = generator(z).view(num_samples, 1, img_size, img_size)

        for j in range(num_samples):
            image_path = os.path.join(epoch_folder, f"image_{j+1}.png")
            save_image(generated_images[j], image_path, normalize=True)

        print(f"Saved images for {epoch}th epoch at: {epoch_folder}")

    # Save model checkpoint, optimizer, and scheduler states every 10 epochs
    if epoch % 10 == 0:
        generator_path = os.path.join(save_dir, f"generator_epoch_{epoch}.pth")
        discriminator_path = os.path.join(save_dir, f"discriminator_epoch_{epoch}.pth")
        optimizer_G_path = os.path.join(save_dir, f"optimizer_G_epoch_{epoch}.pth")
        optimizer_D_path = os.path.join(save_dir, f"optimizer_D_epoch_{epoch}.pth")
        scheduler_G_path = os.path.join(save_dir, f"scheduler_G_epoch_{epoch}.pth")
        scheduler_D_path = os.path.join(save_dir, f"scheduler_D_epoch_{epoch}.pth")

        # Save model state dictionaries
        torch.save(generator.state_dict(), generator_path)
        torch.save(discriminator.state_dict(), discriminator_path)

        # Save optimizer state dictionaries
        torch.save(optimizer_G.state_dict(), optimizer_G_path)
        torch.save(optimizer_D.state_dict(), optimizer_D_path)

        # Save scheduler state dictionaries
        torch.save(scheduler_G.state_dict(), scheduler_G_path)
        torch.save(scheduler_D.state_dict(), scheduler_D_path)

        print(f"Checkpoint saved at epoch {epoch} to Google Drive:\n{generator_path}\n{discriminator_path}")

# -----------------------
#  Save Final Model to Drive
# -----------------------
generator_path = os.path.join(save_dir, "generator_final.pth")
discriminator_path = os.path.join(save_dir, "discriminator_final.pth")
optimizer_G_path = os.path.join(save_dir, "optimizer_G_final.pth")
optimizer_D_path = os.path.join(save_dir, "optimizer_D_final.pth")
scheduler_G_path = os.path.join(save_dir, "scheduler_G_final.pth")
scheduler_D_path = os.path.join(save_dir, "scheduler_D_final.pth")

# Save final model weights
torch.save(generator.state_dict(), generator_path)

# Save final optimizer and scheduler states
torch.save(optimizer_G.state_dict(), optimizer_G_path)
torch.save(optimizer_D.state_dict(), optimizer_D_path)
torch.save(scheduler_G.state_dict(), scheduler_G_path)
torch.save(scheduler_D.state_dict(), scheduler_D_path)

print(f"Final model saved to Google Drive at:\n{generator_path}\n{discriminator_path}")

# ----------------------------
#  Generate and Save 1000 Images to Drive
# ----------------------------
num_samples = 1000  # Number of final images to save
z = torch.randn(num_samples, latent_dim, device=device)  # Generate latent vectors
final_images = generator(z).view(num_samples, 1, img_size, img_size)  # Reshape output

for i in range(num_samples):
    save_path = os.path.join(base_image_dir, f"generated_image_{i+1}.png")
    save_image(final_images[i], save_path, normalize=True)

print(f"Final generated images saved to Google Drive at: {base_image_dir}")

In [ ]:
import torch
from torch.autograd import Variable
import numpy as np
import os
from torchvision.utils import save_image

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define directories
base_image_dir = "/content/drive/My Drive/GAN_Generated_Images"
save_dir = "/content/drive/My Drive/GAN_Weights"
os.makedirs(base_image_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

checkpoint_interval = 10  # Save images every 10 epochs

# Gradient penalty function
def compute_gradient_penalty(D, real_samples, fake_samples):
    alpha = torch.tensor(np.random.random((real_samples.size(0), 1, 1, 1)), dtype=torch.float32, device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = D(interpolates)
    fake = Variable(torch.ones(real_samples.shape[0], 1, dtype=torch.float32, device=device), requires_grad=False)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Image shape
img_shape = (1, img_size, img_size)

# Instantiate models
generator = Generator(latent_dim=latent_dim, img_shape=img_shape).to(device)
discriminator = Discriminator(img_shape=img_shape).to(device)

# Define optimizers
optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr_g, betas=(b1, b2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr_d, betas=(b1, b2))

# Learning rate scheduler
scheduler_G = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_G, T_max=n_epochs)
scheduler_D = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_D, T_max=n_epochs)

# -------- Load Checkpoints (Resume Training) -------- #
resume_epoch = 90  # Change this to the last completed epoch

generator_path = os.path.join(save_dir, f"generator_epoch_{resume_epoch}.pth")
discriminator_path = os.path.join(save_dir, f"discriminator_epoch_{resume_epoch}.pth")
optimizer_G_path = os.path.join(save_dir, f"optimizer_G_epoch_{resume_epoch}.pth")
optimizer_D_path = os.path.join(save_dir, f"optimizer_D_epoch_{resume_epoch}.pth")
scheduler_G_path = os.path.join(save_dir, f"scheduler_G_epoch_{resume_epoch}.pth")
scheduler_D_path = os.path.join(save_dir, f"scheduler_D_epoch_{resume_epoch}.pth")

if os.path.exists(generator_path) and os.path.exists(discriminator_path):
    generator.load_state_dict(torch.load(generator_path, map_location=device))
    discriminator.load_state_dict(torch.load(discriminator_path, map_location=device))
    optimizer_G.load_state_dict(torch.load(optimizer_G_path, map_location=device))
    optimizer_D.load_state_dict(torch.load(optimizer_D_path, map_location=device))
    scheduler_G.load_state_dict(torch.load(scheduler_G_path, map_location=device))
    scheduler_D.load_state_dict(torch.load(scheduler_D_path, map_location=device))
    print(f"Loaded checkpoint from epoch {resume_epoch}. Resuming training...")

# ----------
#  Resume Training
# ----------
batches_done = 0
d_loss_val = 0
g_loss_val = 0

for epoch in range(resume_epoch + 1, n_epochs + 1):
    if epoch % 5 == 1:
        generate_and_display_images(generator, num_images=7, latent_dim=latent_dim, nrow=7, device=device)

    for i, imgs in enumerate(dataloader):
        real_imgs = Variable(imgs.type(torch.float32).to(device))

        # Train Discriminator
        optimizer_D.zero_grad()
        z = Variable(torch.tensor(np.random.normal(0, 1, (imgs.shape[0], latent_dim)), dtype=torch.float32).to(device))
        fake_imgs = generator(z).view(imgs.shape[0], 1, img_size, img_size)
        real_validity = discriminator(real_imgs)
        fake_validity = discriminator(fake_imgs)
        gradient_penalty = compute_gradient_penalty(discriminator, real_imgs.data, fake_imgs.data)
        d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
        d_loss.backward()
        d_loss_val = d_loss.item()
        optimizer_D.step()

        optimizer_G.zero_grad()

        # Train the Generator every n_critic steps
        if i % n_critic == 0:
            fake_imgs = generator(z).view(imgs.shape[0], 1, img_size, img_size)
            fake_validity = discriminator(fake_imgs)
            g_loss = -torch.mean(fake_validity)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
            g_loss.backward()
            optimizer_G.step()
            g_loss_val = g_loss.item()
            if batches_done % sample_interval == 0:
                save_image(fake_imgs.data[:25], "images/%d.png" % batches_done, nrow=5, normalize=True)
            batches_done += 1

        # Step the schedulers (learning rate adjustment)
        scheduler_G.step()
        scheduler_D.step()

    print(f"[Epoch {epoch}/{n_epochs}] [D loss: {d_loss_val}] [G loss: {g_loss_val}]")

    # ----------------------------
    #  Save Images Every 10 Epochs
    # ----------------------------
    if epoch % checkpoint_interval == 0:
        epoch_folder = os.path.join(base_image_dir, f"{epoch}th_epoch")
        os.makedirs(epoch_folder, exist_ok=True)

        num_samples = 25
        z = torch.randn(num_samples, latent_dim, device=device)
        generated_images = generator(z).view(num_samples, 1, img_size, img_size)

        for j in range(num_samples):
            image_path = os.path.join(epoch_folder, f"image_{j+1}.png")
            save_image(generated_images[j], image_path, normalize=True)

        print(f"Saved images for {epoch}th epoch at: {epoch_folder}")

    # Save model checkpoint, optimizer, and scheduler states every 10 epochs
    if epoch % 10 == 0:
        generator_path = os.path.join(save_dir, f"generator_epoch_{epoch}.pth")
        discriminator_path = os.path.join(save_dir, f"discriminator_epoch_{epoch}.pth")
        optimizer_G_path = os.path.join(save_dir, f"optimizer_G_epoch_{epoch}.pth")
        optimizer_D_path = os.path.join(save_dir, f"optimizer_D_epoch_{epoch}.pth")
        scheduler_G_path = os.path.join(save_dir, f"scheduler_G_epoch_{epoch}.pth")
        scheduler_D_path = os.path.join(save_dir, f"scheduler_D_epoch_{epoch}.pth")

        torch.save(generator.state_dict(), generator_path)
        torch.save(discriminator.state_dict(), discriminator_path)
        torch.save(optimizer_G.state_dict(), optimizer_G_path)
        torch.save(optimizer_D.state_dict(), optimizer_D_path)
        torch.save(scheduler_G.state_dict(), scheduler_G_path)
        torch.save(scheduler_D.state_dict(), scheduler_D_path)

        print(f"Checkpoint saved at epoch {epoch} to Google Drive:\n{generator_path}\n{discriminator_path}")

# -----------------------
#  Save Final Model to Drive
# -----------------------
generator_path = os.path.join(save_dir, "generator_final.pth")
discriminator_path = os.path.join(save_dir, "discriminator_final.pth")
optimizer_G_path = os.path.join(save_dir, "optimizer_G_final.pth")
optimizer_D_path = os.path.join(save_dir, "optimizer_D_final.pth")
scheduler_G_path = os.path.join(save_dir, "scheduler_G_final.pth")
scheduler_D_path = os.path.join(save_dir, "scheduler_D_final.pth")

torch.save(generator.state_dict(), generator_path)
torch.save(discriminator.state_dict(), discriminator_path)
torch.save(optimizer_G.state_dict(), optimizer_G_path)
torch.save(optimizer_D.state_dict(), optimizer_D_path)
torch.save(scheduler_G.state_dict(), scheduler_G_path)
torch.save(scheduler_D.state_dict(), scheduler_D_path)

print(f"Final model saved to Google Drive at:\n{generator_path}\n{discriminator_path}")

In [ ]:
import torch
from torch.autograd import Variable
import numpy as np
import os
from torchvision.utils import save_image

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define directories
base_image_dir = "/content/drive/My Drive/GAN_Generated_Images"
save_dir = "/content/drive/My Drive/GAN_Weights"
final_image_dir = os.path.join(base_image_dir, "Final_Generated_Images")

os.makedirs(base_image_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)
os.makedirs(final_image_dir, exist_ok=True)

checkpoint_interval = 10  # Save images every 10 epochs

# Gradient penalty function
def compute_gradient_penalty(D, real_samples, fake_samples):
    alpha = torch.tensor(np.random.random((real_samples.size(0), 1, 1, 1)), dtype=torch.float32, device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = D(interpolates)
    fake = Variable(torch.ones(real_samples.shape[0], 1, dtype=torch.float32, device=device), requires_grad=False)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Image shape
img_shape = (1, img_size, img_size)

# Instantiate models
generator = Generator(latent_dim=latent_dim, img_shape=img_shape).to(device)
discriminator = Discriminator(img_shape=img_shape).to(device)

# Define optimizers
optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr_g, betas=(b1, b2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr_d, betas=(b1, b2))

# Learning rate scheduler
scheduler_G = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_G, T_max=n_epochs)
scheduler_D = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_D, T_max=n_epochs)

# -------- Load Checkpoints (Resume Training) -------- #
resume_epoch = 190  # Change this to the last completed epoch
n_epochs = resume_epoch + 10  # Extend training by 5 more epochs

generator_path = os.path.join(save_dir, f"generator_epoch_{resume_epoch}.pth")
discriminator_path = os.path.join(save_dir, f"discriminator_epoch_{resume_epoch}.pth")
optimizer_G_path = os.path.join(save_dir, f"optimizer_G_epoch_{resume_epoch}.pth")
optimizer_D_path = os.path.join(save_dir, f"optimizer_D_epoch_{resume_epoch}.pth")
scheduler_G_path = os.path.join(save_dir, f"scheduler_G_epoch_{resume_epoch}.pth")
scheduler_D_path = os.path.join(save_dir, f"scheduler_D_epoch_{resume_epoch}.pth")

if os.path.exists(generator_path) and os.path.exists(discriminator_path):
    generator.load_state_dict(torch.load(generator_path, map_location=device))
    discriminator.load_state_dict(torch.load(discriminator_path, map_location=device))
    optimizer_G.load_state_dict(torch.load(optimizer_G_path, map_location=device))
    optimizer_D.load_state_dict(torch.load(optimizer_D_path, map_location=device))
    scheduler_G.load_state_dict(torch.load(scheduler_G_path, map_location=device))
    scheduler_D.load_state_dict(torch.load(scheduler_D_path, map_location=device))
    print(f"Loaded checkpoint from epoch {resume_epoch}. Resuming training...")

# ----------
#  Resume Training
# ----------
batches_done = 0
d_loss_val = 0
g_loss_val = 0

for epoch in range(resume_epoch + 1, n_epochs + 1):
    if epoch % 5 == 1:
        generate_and_display_images(generator, num_images=7, latent_dim=latent_dim, nrow=7, device=device)

    for i, imgs in enumerate(dataloader):
        real_imgs = Variable(imgs.type(torch.float32).to(device))

        # Train Discriminator
        optimizer_D.zero_grad()
        z = Variable(torch.tensor(np.random.normal(0, 1, (imgs.shape[0], latent_dim)), dtype=torch.float32).to(device))
        fake_imgs = generator(z).view(imgs.shape[0], 1, img_size, img_size)
        real_validity = discriminator(real_imgs)
        fake_validity = discriminator(fake_imgs)
        gradient_penalty = compute_gradient_penalty(discriminator, real_imgs.data, fake_imgs.data)
        d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
        d_loss.backward()
        d_loss_val = d_loss.item()
        optimizer_D.step()

        optimizer_G.zero_grad()

        # Train the Generator every n_critic steps
        if i % n_critic == 0:
            fake_imgs = generator(z).view(imgs.shape[0], 1, img_size, img_size)
            fake_validity = discriminator(fake_imgs)
            g_loss = -torch.mean(fake_validity)
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
            g_loss.backward()
            optimizer_G.step()
            g_loss_val = g_loss.item()
            batches_done += 1

        # Step the schedulers (learning rate adjustment)
        scheduler_G.step()
        scheduler_D.step()

    print(f"[Epoch {epoch}/{n_epochs}] [D loss: {d_loss_val}] [G loss: {g_loss_val}]")

    # ----------------------------
    #  Save Model Checkpoints
    # ----------------------------
    if epoch % checkpoint_interval == 0:
        generator_path = os.path.join(save_dir, f"generator_epoch_{epoch}.pth")
        discriminator_path = os.path.join(save_dir, f"discriminator_epoch_{epoch}.pth")
        optimizer_G_path = os.path.join(save_dir, f"optimizer_G_epoch_{epoch}.pth")
        optimizer_D_path = os.path.join(save_dir, f"optimizer_D_epoch_{epoch}.pth")
        scheduler_G_path = os.path.join(save_dir, f"scheduler_G_epoch_{epoch}.pth")
        scheduler_D_path = os.path.join(save_dir, f"scheduler_D_epoch_{epoch}.pth")

        torch.save(generator.state_dict(), generator_path)
        torch.save(discriminator.state_dict(), discriminator_path)
        torch.save(optimizer_G.state_dict(), optimizer_G_path)
        torch.save(optimizer_D.state_dict(), optimizer_D_path)
        torch.save(scheduler_G.state_dict(), scheduler_G_path)
        torch.save(scheduler_D.state_dict(), scheduler_D_path)

        print(f"Checkpoint saved at epoch {epoch} to Google Drive.")

# -----------------------
#  Generate 2000 Images After Training
# -----------------------
num_final_images = 2000
batch_size = 100  # Generate in batches of 100 to manage memory
num_batches = num_final_images // batch_size

print(f"Generating {num_final_images} final images...")

for batch in range(num_batches):
    z = torch.randn(batch_size, latent_dim, device=device)
    generated_images = generator(z).view(batch_size, 1, img_size, img_size)

    for j in range(batch_size):
        image_path = os.path.join(final_image_dir, f"image_{batch * batch_size + j + 1}.png")
        save_image(generated_images[j], image_path, normalize=True)

print(f"All {num_final_images} images saved to: {final_image_dir}")

# -----------------------
#  Save Final Model to Drive
# -----------------------
generator_path = os.path.join(save_dir, "generator_final.pth")
discriminator_path = os.path.join(save_dir, "discriminator_final.pth")
optimizer_G_path = os.path.join(save_dir, "optimizer_G_final.pth")
optimizer_D_path = os.path.join(save_dir, "optimizer_D_final.pth")
scheduler_G_path = os.path.join(save_dir, "scheduler_G_final.pth")
scheduler_D_path = os.path.join(save_dir, "scheduler_D_final.pth")

torch.save(generator.state_dict(), generator_path)
torch.save(discriminator.state_dict(), discriminator_path)
torch.save(optimizer_G.state_dict(), optimizer_G_path)
torch.save(optimizer_D.state_dict(), optimizer_D_path)
torch.save(scheduler_G.state_dict(), scheduler_G_path)
torch.save(scheduler_D.state_dict(), scheduler_D_path)

print(f"Final model saved to Google Drive at:\n{generator_path}\n{discriminator_path}")